## 06 — Baseline Hough com protocolo congelado (v3)
O notebook só inicia após a aprovação formal do dataset v3. Detector, NMS e thresholds
geométricos são escolhidos exclusivamente na validação. O teste possui trava persistente
no Drive e não roda por padrão.
São reportados separadamente: IoU, geometria de centerline e critério combinado.

In [1]:
%pip install -q opencv-python-headless pandas tqdm

from google.colab import drive
from pathlib import Path
import hashlib, json, math, shutil, time
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}", flush=True)

def copiar_arvore_com_progresso(origem, destino, descricao):
    origem, destino = Path(origem), Path(destino)
    arquivos = sorted(p for p in origem.rglob('*') if p.is_file())
    log_etapa(f'{descricao}: início ({len(arquivos)} arquivos)')
    inicio = time.perf_counter()
    for arquivo in tqdm(arquivos, desc=descricao, unit='arq'):
        alvo = destino / arquivo.relative_to(origem)
        alvo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(arquivo, alvo)
    log_etapa(f'{descricao}: concluído em {time.perf_counter()-inicio:.1f}s')

drive.mount('/content/drive', force_remount=True)
BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
RUN_ID=(BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt').read_text().strip()
DRIVE_RUN=BACKUP_DIR/'data/synthetic_runs'/RUN_ID
RUN=Path('data/synthetic_runs')/RUN_ID
copiar_arvore_com_progresso(DRIVE_RUN, RUN, 'Restaurando RUN para o Hough')
validacao=json.loads((RUN/'resumo_validacao_v3.json').read_text())
assert validacao.get('aprovado_para_treino') is True,'Dataset v3 ainda não foi aprovado no notebook 05.'
df=pd.read_csv(RUN/'manifest_geracao_v3.csv')
HASH_MANIFEST=hashlib.sha256((RUN/'manifest_geracao_v3.csv').read_bytes()).hexdigest()
assert HASH_MANIFEST==validacao['sha256_manifest']
HOUGH=RUN/'hough_v3'; HOUGH.mkdir(parents=True,exist_ok=True)
print(RUN_ID,HASH_MANIFEST)

Mounted at /content/drive
[10:10:49] Restaurando RUN para o Hough: início (12018 arquivos)


Restaurando RUN para o Hough:   0%|          | 0/12018 [00:00<?, ?arq/s]

[11:54:54] Restaurando RUN para o Hough: concluído em 6245.1s
synthetic_v3_c41d5091de56 b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42


## 1. Detector e NMS no espaço geométrico de linhas

In [2]:
def parametros_linha(p0,p1):
    p0,p1=np.array(p0,dtype=float),np.array(p1,dtype=float)
    vetor=p1-p0; comp=float(np.linalg.norm(vetor)); direcao=vetor/comp if comp>0 else np.array([1.,0.])
    return (p0+p1)/2,direcao,comp

def erro_angular(d1,d2): return float(np.degrees(np.arccos(np.clip(abs(np.dot(d1,d2)),-1,1))))

def distancia_ponto_linha(ponto,centro,direcao):
    v=np.asarray(ponto)-np.asarray(centro)
    return float(np.linalg.norm(v-np.dot(v,direcao)*direcao))

def cobertura_projetada(p0a,p1a,p0b,p1b):
    cb,db,lb=parametros_linha(p0b,p1b)
    if lb<=0:return 0.0
    t=sorted([np.dot(np.asarray(p0a)-cb,db),np.dot(np.asarray(p1a)-cb,db)])
    inter=max(0,min(t[1],lb/2)-max(t[0],-lb/2))
    return float(inter/lb)

def detectar_hough(img,pre_blur_sigma,canny_lo,canny_hi,hough_threshold,min_length_frac,max_gap):
    proc=cv2.GaussianBlur(img,(0,0),pre_blur_sigma) if pre_blur_sigma>0 else img
    bordas=cv2.Canny(proc,canny_lo,canny_hi)
    min_len=int(math.hypot(*img.shape[:2])*min_length_frac)
    linhas=cv2.HoughLinesP(bordas,1,np.pi/180,hough_threshold,minLineLength=min_len,maxLineGap=max_gap)
    if linhas is None:return []
    # Compatível com retornos (N,1,4) e (N,4) do OpenCV.
    segmentos=np.asarray(linhas).reshape(-1,4)
    return [((int(x1),int(y1)),(int(x2),int(y2))) for x1,y1,x2,y2 in segmentos]

def nms_linhas(linhas,angulo_max=3.0,dist_max=12.0,cobertura_min=0.30):
    ordenadas=sorted(linhas,key=lambda l:-math.dist(l[0],l[1])); mantidas=[]
    for atual in ordenadas:
        ca,da,_=parametros_linha(*atual); suprimir=False
        for mantida in mantidas:
            cm,dm,_=parametros_linha(*mantida)
            if erro_angular(da,dm)>angulo_max:continue
            if max(distancia_ponto_linha(ca,cm,dm),distancia_ponto_linha(cm,ca,da))>dist_max:continue
            cob=max(cobertura_projetada(*atual,*mantida),cobertura_projetada(*mantida,*atual))
            if cob>=cobertura_min:suprimir=True;break
        if not suprimir:mantidas.append(atual)
    return mantidas

def linha_para_obb(p0,p1,espessura,w,h):
    x0,y0=p0;x1,y1=p1;dx,dy=x1-x0,y1-y0;comp=math.hypot(dx,dy) or 1e-6
    px,py=-dy/comp,dx/comp;m=espessura/2
    return [((x0+px*m)/w,(y0+py*m)/h),((x1+px*m)/w,(y1+py*m)/h),
            ((x1-px*m)/w,(y1-py*m)/h),((x0-px*m)/w,(y0-py*m)/h)]

def iou_obb(a,b,w,h):
    pa=np.array([(x*w,y*h) for x,y in a],np.int32);pb=np.array([(x*w,y*h) for x,y in b],np.int32)
    ma=np.zeros((h,w),np.uint8);mb=np.zeros((h,w),np.uint8)
    cv2.fillPoly(ma,[pa],1);cv2.fillPoly(mb,[pb],1)
    inter=np.logical_and(ma,mb).sum();uniao=np.logical_or(ma,mb).sum()
    return float(inter/uniao) if uniao else 0.0

def ler_obb(caminho):
    t=Path(caminho).read_text().split()
    if not t:return None
    v=[float(x) for x in t[1:9]]
    return [(v[i],v[i+1]) for i in range(0,8,2)]

def obb_para_centerline(obb,w,h):
    p=np.array([(x*w,y*h) for x,y in obb]);d01=np.linalg.norm(p[0]-p[1]);d12=np.linalg.norm(p[1]-p[2])
    if d01>=d12:return tuple((p[0]+p[3])/2),tuple((p[1]+p[2])/2)
    return tuple((p[0]+p[1])/2),tuple((p[2]+p[3])/2)

## 2. Métricas sem atalhos artificiais
Ângulo, distância e cobertura são calculados para todo candidato, inclusive quando o IoU
já ultrapassa o limiar. Assim, as médias finais representam medições reais.

In [3]:
def metricas_candidato(linha,gt,w,h,espessura_match=3):
    p0,p1=linha;obb=linha_para_obb(p0,p1,espessura_match,w,h)
    g0,g1=obb_para_centerline(gt,w,h);cc,dc,_=parametros_linha(p0,p1);cg,dg,_=parametros_linha(g0,g1)
    return {'p0':p0,'p1':p1,'iou':iou_obb(obb,gt,w,h),'angulo':erro_angular(dc,dg),
            'distancia':distancia_ponto_linha(cc,cg,dg),'cobertura':cobertura_projetada(p0,p1,g0,g1)}

def escolher_match(metricas,limiares,modo):
    candidatos=[]
    for m in metricas:
        miou=m['iou']>=limiares['iou']
        mgeo=(m['angulo']<=limiares['angulo'] and m['distancia']<=limiares['distancia'] and
              m['cobertura']>=limiares['cobertura'])
        bate={'iou':miou,'geometria':mgeo,'combinado':miou or mgeo}[modo]
        if not bate:continue
        score_iou=m['iou']/limiares['iou'] if miou else -np.inf
        score_geo=((1-m['angulo']/limiares['angulo'])+(1-m['distancia']/limiares['distancia'])+m['cobertura']) if mgeo else -np.inf
        score={'iou':score_iou,'geometria':score_geo,'combinado':max(score_iou,score_geo)}[modo]
        candidatos.append((score,m))
    return max(candidatos,key=lambda x:x[0])[1] if candidatos else None

# Testes unitários mínimos
assert abs(erro_angular(np.array([1.,0.]),np.array([-1.,0.])))<1e-9
assert abs(erro_angular(np.array([1.,0.]),np.array([0.,1.]))-90)<1e-9
assert abs(cobertura_projetada((0,0),(50,0),(0,0),(100,0))-.5)<1e-9
assert abs(distancia_ponto_linha((0,10),(0,0),np.array([1.,0.]))-10)<1e-9
print('Testes geométricos aprovados.')

Testes geométricos aprovados.


## 3. Avaliação por imagem e conjunto

In [4]:
def avaliar_imagem(linha,detector,limiares):
    imgp=RUN/'images'/linha.split/(linha.id+'.png');lblp=RUN/'labels'/linha.split/(linha.id+'.txt')
    img=cv2.imread(str(imgp),cv2.IMREAD_GRAYSCALE);h,w=img.shape
    brutas=detectar_hough(img,**detector);candidatos=nms_linhas(brutas)
    gt=ler_obb(lblp)
    base={'id':linha.id,'split':linha.split,'tipo':linha.tipo,'subtipo':linha.subtipo,
          'fundo_origem':linha.fundo_origem,'sha256_fundo':linha.sha256_fundo,
          'fonte_fundo':linha.fonte_fundo,'n_candidatos':len(candidatos)}
    if gt is None:
        for modo in ['iou','geometria','combinado']:
            base[f'acertou_{modo}']=False;base[f'tp_{modo}']=0;base[f'fn_{modo}']=0;base[f'fp_{modo}']=len(candidatos)
        base.update({'angulo_match':np.nan,'distancia_match':np.nan,'cobertura_match':np.nan,'iou_match':np.nan})
        return base
    mets=[metricas_candidato(c,gt,w,h) for c in candidatos]
    matches={modo:escolher_match(mets,limiares,modo) for modo in ['iou','geometria','combinado']}
    for modo,m in matches.items():
        base[f'acertou_{modo}']=m is not None;base[f'tp_{modo}']=int(m is not None)
        base[f'fn_{modo}']=int(m is None);base[f'fp_{modo}']=max(0,len(candidatos)-int(m is not None))
    melhor=matches['combinado']
    base.update({'angulo_match':melhor['angulo'] if melhor else np.nan,
                 'distancia_match':melhor['distancia'] if melhor else np.nan,
                 'cobertura_match':melhor['cobertura'] if melhor else np.nan,
                 'iou_match':melhor['iou'] if melhor else np.nan})
    return base

def resumir(resultados,modo):
    tp=int(resultados[f'tp_{modo}'].sum());fp=int(resultados[f'fp_{modo}'].sum());fn=int(resultados[f'fn_{modo}'].sum())
    p=tp/(tp+fp) if tp+fp else 0;r=tp/(tp+fn) if tp+fn else 0;f1=2*p*r/(p+r) if p+r else 0
    neg=resultados[resultados.tipo=='negativo']; taxa=float((neg[f'fp_{modo}']>0).mean()) if len(neg) else 0
    return {'modo':modo,'TP':tp,'FP':fp,'FN':fn,'precisao':p,'recall':r,'f1':f1,
            'fp_por_imagem_negativa':float(neg[f'fp_{modo}'].mean()) if len(neg) else 0,
            'taxa_imagens_negativas_com_fp':taxa}

def avaliar_conjunto(dados,detector,limiares,descricao):
    linhas=[avaliar_imagem(l,detector,limiares) for l in tqdm(dados.itertuples(),total=len(dados),desc=descricao)]
    resultados=pd.DataFrame(linhas)
    resumos=pd.DataFrame([resumir(resultados,m) for m in ['iou','geometria','combinado']])
    return resultados,resumos

## 4. Calibração exclusiva na validação
Distância geométrica é limitada a 6–14 px, em vez de 5% da diagonal (~36 px).

In [5]:
DETECTORES=[
  {'nome':'permissivo','pre_blur_sigma':0,'canny_lo':50,'canny_hi':150,'hough_threshold':40,'min_length_frac':.15,'max_gap':15},
  {'nome':'rigoroso','pre_blur_sigma':0,'canny_lo':50,'canny_hi':150,'hough_threshold':90,'min_length_frac':.25,'max_gap':8},
  {'nome':'pre_desfoque','pre_blur_sigma':2,'canny_lo':50,'canny_hi':150,'hough_threshold':70,'min_length_frac':.20,'max_gap':20},
]
MATCHERS=[
  {'nome_match':'estrito','iou':.30,'angulo':5.0,'distancia':6.0,'cobertura':.70},
  {'nome_match':'equilibrado','iou':.30,'angulo':10.0,'distancia':10.0,'cobertura':.60},
  {'nome_match':'amplo','iou':.30,'angulo':15.0,'distancia':14.0,'cobertura':.50},
]

df_val=df[df.split=='val'].copy()
grupos=df_val.groupby(['tipo','subtipo']);partes=[]
for _,g in grupos:
    n=max(1,round(300*len(g)/len(df_val)));partes.append(g.sample(min(n,len(g)),random_state=7))
amostra=pd.concat(partes).drop_duplicates('id').head(300)
calibracoes=[]
for det in DETECTORES:
    nome_det=det['nome'];params_det={k:v for k,v in det.items() if k!='nome'}
    for matcher in MATCHERS:
        resultados,resumos=avaliar_conjunto(amostra,params_det,matcher,f'{nome_det}/{matcher["nome_match"]}')
        combinado=resumos[resumos.modo=='combinado'].iloc[0].to_dict()
        calibracoes.append({'detector':nome_det,**params_det,**matcher,**{f'metrica_{k}':v for k,v in combinado.items() if k!='modo'}})
df_cal=pd.DataFrame(calibracoes).sort_values(['metrica_f1','metrica_precisao'],ascending=False)
vencedor=df_cal.iloc[0]
print(df_cal[['detector','nome_match','metrica_precisao','metrica_recall','metrica_f1']])

permissivo/estrito:   0%|          | 0/300 [00:00<?, ?it/s]

permissivo/equilibrado:   0%|          | 0/300 [00:00<?, ?it/s]

permissivo/amplo:   0%|          | 0/300 [00:00<?, ?it/s]

rigoroso/estrito:   0%|          | 0/300 [00:00<?, ?it/s]

rigoroso/equilibrado:   0%|          | 0/300 [00:00<?, ?it/s]

rigoroso/amplo:   0%|          | 0/300 [00:00<?, ?it/s]

pre_desfoque/estrito:   0%|          | 0/300 [00:00<?, ?it/s]

pre_desfoque/equilibrado:   0%|          | 0/300 [00:00<?, ?it/s]

pre_desfoque/amplo:   0%|          | 0/300 [00:00<?, ?it/s]

       detector   nome_match  metrica_precisao  metrica_recall  metrica_f1
5      rigoroso        amplo          0.821918        0.400000    0.538117
4      rigoroso  equilibrado          0.712329        0.346667    0.466368
3      rigoroso      estrito          0.643836        0.313333    0.421525
8  pre_desfoque        amplo          0.760000        0.253333    0.380000
7  pre_desfoque  equilibrado          0.740000        0.246667    0.370000
2    permissivo        amplo          0.276224        0.526667    0.362385
1    permissivo  equilibrado          0.265734        0.506667    0.348624
0    permissivo      estrito          0.248252        0.473333    0.325688
6  pre_desfoque      estrito          0.640000        0.213333    0.320000


## 5. Congelamento e validação completa

In [6]:
chaves_det=['pre_blur_sigma','canny_lo','canny_hi','hough_threshold','min_length_frac','max_gap']
chaves_match=['iou','angulo','distancia','cobertura']
DETECTOR_FINAL={k:(float(vencedor[k]) if k in ['pre_blur_sigma','min_length_frac'] else int(vencedor[k])) for k in chaves_det}
MATCHER_FINAL={k:float(vencedor[k]) for k in chaves_match}
congelado={'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,'detector':DETECTOR_FINAL,
           'matcher':MATCHER_FINAL,'selecionado_somente_em':'val','amostra_val_n':len(amostra)}
texto=json.dumps(congelado,sort_keys=True)
congelado['sha256_config']=hashlib.sha256(texto.encode()).hexdigest()
(HOUGH/'config_congelada_v3.json').write_text(json.dumps(congelado,indent=2),encoding='utf-8')
df_cal.to_csv(HOUGH/'calibracao_v3.csv',index=False)

resultados_val,resumos_val=avaliar_conjunto(df_val,DETECTOR_FINAL,MATCHER_FINAL,'Validação completa')
resultados_val.to_csv(HOUGH/'resultado_val_v3.csv',index=False);resumos_val.to_csv(HOUGH/'resumo_val_v3.csv',index=False)
print(json.dumps(congelado,indent=2));print(resumos_val)
copiar_arvore_com_progresso(HOUGH, DRIVE_RUN/'hough_v3', 'Salvando calibração Hough')

Validação completa:   0%|          | 0/900 [00:00<?, ?it/s]

{
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "detector": {
    "pre_blur_sigma": 0.0,
    "canny_lo": 50,
    "canny_hi": 150,
    "hough_threshold": 90,
    "min_length_frac": 0.25,
    "max_gap": 8
  },
  "matcher": {
    "iou": 0.3,
    "angulo": 15.0,
    "distancia": 14.0,
    "cobertura": 0.5
  },
  "selecionado_somente_em": "val",
  "amostra_val_n": 300,
  "sha256_config": "14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5"
}
        modo   TP  FP   FN  precisao    recall        f1  \
0        iou  103  89  347  0.536458  0.228889  0.320872   
1  geometria  163  29  287  0.848958  0.362222  0.507788   
2  combinado  163  29  287  0.848958  0.362222  0.507788   

   fp_por_imagem_negativa  taxa_imagens_negativas_com_fp  
0                0.026667                           0.02  
1                0.026667                           0.02  
2                0.026667                

Salvando calibração Hough:   0%|          | 0/4 [00:00<?, ?arq/s]

[11:58:45] Salvando calibração Hough: concluído em 0.1s


## 6. Teste final protegido
Revise a configuração congelada e a validação. Depois altere RODAR_TESTE_FINAL para
True uma única vez. Se já existir resultado no Drive, a célula recusa nova execução.

In [7]:
RODAR_TESTE_FINAL=True
resultado_drive=DRIVE_RUN/'hough_v3/resultado_test_v3.csv'
if resultado_drive.exists():
    print('Teste final já existe e permanece congelado:',resultado_drive)
elif RODAR_TESTE_FINAL:
    df_test=df[df.split=='test'].copy()
    resultados_test,resumos_test=avaliar_conjunto(df_test,DETECTOR_FINAL,MATCHER_FINAL,'TESTE FINAL v3')
    resultados_test.to_csv(HOUGH/'resultado_test_v3.csv',index=False)
    resumos_test.to_csv(HOUGH/'resumo_test_v3.csv',index=False)
    trava={'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,'sha256_config':congelado['sha256_config'],
           'sha256_resultado':hashlib.sha256((HOUGH/'resultado_test_v3.csv').read_bytes()).hexdigest()}
    (HOUGH/'TRAVA_TESTE_FINAL.json').write_text(json.dumps(trava,indent=2),encoding='utf-8')
    copiar_arvore_com_progresso(HOUGH, DRIVE_RUN/'hough_v3', 'Salvando teste final Hough')
    print(resumos_test)
else:
    print('Teste não executado. Altere RODAR_TESTE_FINAL somente após congelar o protocolo.')

TESTE FINAL v3:   0%|          | 0/900 [00:00<?, ?it/s]

[12:06:29] Salvando teste final Hough: início (7 arquivos)


Salvando teste final Hough:   0%|          | 0/7 [00:00<?, ?arq/s]

[12:06:29] Salvando teste final Hough: concluído em 0.1s
        modo   TP   FP   FN  precisao    recall        f1  \
0        iou   87  248  363  0.259701  0.193333  0.221656   
1  geometria  148  187  302  0.441791  0.328889  0.377070   
2  combinado  148  187  302  0.441791  0.328889  0.377070   

   fp_por_imagem_negativa  taxa_imagens_negativas_com_fp  
0                0.204444                       0.044444  
1                0.204444                       0.044444  
2                0.204444                       0.044444  


In [8]:
print((HOUGH / 'TRAVA_TESTE_FINAL.json').read_text(encoding='utf-8'))

{
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config": "14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5",
  "sha256_resultado": "5b476de5a6015b6c38758a43e8c46719050bc036d28362d5f7d43f13e8ffa3b8"
}


## 7. Detalhamento e intervalo de confiança agrupado por fundo
Execute após o teste final existir. O bootstrap reamostra fundos, não imagens individuais.

In [10]:
if not (HOUGH/'resultado_test_v3.csv').exists() and resultado_drive.exists():
    shutil.copy2(resultado_drive,HOUGH/'resultado_test_v3.csv')
if (HOUGH/'resultado_test_v3.csv').exists():
    rt=pd.read_csv(HOUGH/'resultado_test_v3.csv')
    recall_subtipo=rt[rt.tipo=='positivo'].groupby('subtipo')['acertou_combinado'].mean()
    print('Recall combinado por modo de brilho:')
    print(recall_subtipo)
    print('\nFP/imagem e taxa de frames com FP:')
    neg=rt[rt.tipo=='negativo']
    fp_subtipo=neg.groupby('subtipo').agg(
        fp_por_imagem=('fp_combinado','mean'),
        taxa_frames_com_fp=('fp_combinado',lambda s:float((s>0).mean()))
    )
    print(fp_subtipo)
    print('\nMétricas geométricas reais entre matches combinados:')
    ac=rt[(rt.tipo=='positivo')&(rt.acertou_combinado)]
    geometria=ac[['angulo_match','distancia_match','cobertura_match','iou_match']].agg(['mean','median'])
    print(geometria)

    por_fundo=rt.groupby('sha256_fundo')[['tp_combinado','fp_combinado','fn_combinado']].sum()
    rng=np.random.default_rng(20260823);f1s=[];ids=por_fundo.index.to_numpy()
    for _ in tqdm(range(2000), desc='Bootstrap por fundo', unit='rep'):
        amostra_ids=rng.choice(ids,size=len(ids),replace=True);s=por_fundo.loc[amostra_ids].sum()
        p=s.tp_combinado/(s.tp_combinado+s.fp_combinado) if s.tp_combinado+s.fp_combinado else 0
        r=s.tp_combinado/(s.tp_combinado+s.fn_combinado) if s.tp_combinado+s.fn_combinado else 0
        f1s.append(2*p*r/(p+r) if p+r else 0)
    ic95=np.percentile(f1s,[2.5,97.5])
    print('IC95% F1 combinado, bootstrap por fundo:',ic95)

    trava=json.loads((HOUGH/'TRAVA_TESTE_FINAL.json').read_text(encoding='utf-8'))
    resumo_detalhado={
        'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
        'sha256_config':trava['sha256_config'],'sha256_resultado':trava['sha256_resultado'],
        'recall_combinado_por_subtipo':{k:float(v) for k,v in recall_subtipo.items()},
        'negativos_por_subtipo':{
            k:{'fp_por_imagem':float(v.fp_por_imagem),
               'taxa_frames_com_fp':float(v.taxa_frames_com_fp)}
            for k,v in fp_subtipo.iterrows()
        },
        'geometria_matches_combinados':{
            coluna:{'media':float(geometria.loc['mean',coluna]),
                    'mediana':float(geometria.loc['median',coluna])}
            for coluna in geometria.columns
        },
        'bootstrap_f1_por_fundo':{
            'seed':20260823,'replicacoes':2000,'n_fundos':int(len(ids)),
            'ic95_percentil':[float(ic95[0]),float(ic95[1])]
        }
    }
    resumo_path=HOUGH/'resumo_detalhado_test_v3.json'
    bootstrap_path=HOUGH/'bootstrap_f1_por_fundo_v3.csv'
    resumo_path.write_text(json.dumps(resumo_detalhado,indent=2),encoding='utf-8')
    pd.DataFrame({'f1_combinado':f1s}).to_csv(bootstrap_path,index=False)
    for arquivo in [resumo_path,bootstrap_path]:
        shutil.copy2(arquivo,DRIVE_RUN/'hough_v3'/arquivo.name)
    print('Resumo detalhado e bootstrap salvos no Drive.')
    print(json.dumps(resumo_detalhado,indent=2,ensure_ascii=False))

Recall combinado por modo de brilho:
subtipo
constante    0.418519
tracejado    0.014925
variavel     0.300885
Name: acertou_combinado, dtype: float64

FP/imagem e taxa de frames com FP:
                fp_por_imagem  taxa_frames_com_fp
subtipo                                          
hard_sintetico       0.333333            0.122222
limpo                0.172222            0.025000

Métricas geométricas reais entre matches combinados:
        angulo_match  distancia_match  cobertura_match  iou_match
mean        0.284997         2.027404         0.830076   0.326635
median      0.194266         2.071902         0.854393   0.316591


Bootstrap por fundo:   0%|          | 0/2000 [00:00<?, ?rep/s]

IC95% F1 combinado, bootstrap por fundo: [0.26664845 0.51242627]
Resumo detalhado e bootstrap salvos no Drive.
{
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_config": "14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5",
  "sha256_resultado": "5b476de5a6015b6c38758a43e8c46719050bc036d28362d5f7d43f13e8ffa3b8",
  "recall_combinado_por_subtipo": {
    "constante": 0.4185185185185185,
    "tracejado": 0.014925373134328358,
    "variavel": 0.3008849557522124
  },
  "negativos_por_subtipo": {
    "hard_sintetico": {
      "fp_por_imagem": 0.3333333333333333,
      "taxa_frames_com_fp": 0.12222222222222222
    },
    "limpo": {
      "fp_por_imagem": 0.17222222222222222,
      "taxa_frames_com_fp": 0.025
    }
  },
  "geometria_matches_combinados": {
    "angulo_match": {
      "media": 0.28499704133526277,
      "mediana": 0.1942660455000658
    },
    "distancia_match": {
      "medi